# Baseline models

In [ ]:
# ============================================================
# SCRIPT 2: STRUCTURED-ONLY ML BASELINE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    log_loss,
    recall_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Choose one:
# "structured_no_odds"
# "odds_only"
# "structured_with_odds"
EXPERIMENT = "structured_with_odds"

BASE_DIR = "/content/drive/MyDrive/Speciale"

PATH_NO_ODDS = os.path.join(BASE_DIR, "final_no_odds_dataset.csv")
PATH_WITH_ODDS = os.path.join(BASE_DIR, "final_with_odds_dataset.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "ml_baseline_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Experiment logic (FIXED)
# ------------------------------------------------------------

if EXPERIMENT == "structured_no_odds":
    USE_STRUCTURED = True
    USE_ODDS = False
    DATA_PATH = PATH_NO_ODDS

elif EXPERIMENT == "odds_only":
    USE_STRUCTURED = False
    USE_ODDS = True
    DATA_PATH = PATH_WITH_ODDS

elif EXPERIMENT == "structured_with_odds":
    USE_STRUCTURED = True
    USE_ODDS = True
    DATA_PATH = PATH_WITH_ODDS

else:
    raise ValueError("Invalid EXPERIMENT")

print("Experiment:", EXPERIMENT)
print("Use structured:", USE_STRUCTURED)
print("Use odds:", USE_ODDS)
print("Data path:", DATA_PATH)

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

df.columns = [
    str(c).replace("\u00a0", "").replace("\n", "").replace("\t", "").strip()
    for c in df.columns
]

df["MatchDatetime"] = pd.to_datetime(df["MatchDatetime"], errors="coerce")

target_map = {"H": 0, "D": 1, "A": 2}
target_names = {0: "Home win", 1: "Draw", 2: "Away win"}

df["target"] = df["FTR"].map(target_map)

if df["target"].isna().any():
    raise ValueError("Some FTR values could not be mapped.")

# ------------------------------------------------------------
# 3. DEFINE STRUCTURED FEATURES
# ------------------------------------------------------------

base_feature_cols = [
    "Home_attack",
    "Home_defence",
    "Away_attack",
    "Away_defence",
    "Home_PPG",
    "Away_PPG",
    "Home_GDpg",
    "Away_GDpg",
    "Home_form_PPG",
    "Away_form_PPG",
    "Attack_diff",
    "Defence_diff",
    "PPG_diff",
    "GDpg_diff",
    "Form_PPG_diff",
    "Interaction_home",
    "Interaction_away",
    "Strength_diff_abs"
]

odds_feature_cols = [
    "AvgH",
    "AvgD",
    "AvgA"
]

structured_feature_cols = []

if USE_STRUCTURED:
    structured_feature_cols += base_feature_cols

if USE_ODDS:
    structured_feature_cols += odds_feature_cols

missing_features = [
    col for col in structured_feature_cols
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")

def clean_numeric_column(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace({"-": np.nan, "": np.nan, "nan": np.nan, "None": np.nan})
        .astype(float)
    )

for col in structured_feature_cols:
    df[col] = clean_numeric_column(df[col])

essential_cols = [
    "MatchDatetime",
    "season",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "target"
] + structured_feature_cols

before_drop = df.shape[0]
df = df.dropna(subset=essential_cols).copy()
after_drop = df.shape[0]

df = df.sort_values("MatchDatetime").reset_index(drop=True)

print("Rows before drop:", before_drop)
print("Rows after drop:", after_drop)
print("Final dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].map(target_names).value_counts())

print("\nStructured features:")
print(structured_feature_cols)

# ------------------------------------------------------------
# 4. SPLIT DATA
# ------------------------------------------------------------

X_structured = df[structured_feature_cols].values
y = df["target"].values

indices = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=RANDOM_STATE
)

df["split"] = ""
df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

X_train_raw = X_structured[train_idx]
X_val_raw = X_structured[val_idx]
X_test_raw = X_structured[test_idx]

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

print("\n--- SPLIT SHAPES ---")
print("Train:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

print("\n--- TARGET DISTRIBUTION ---")
print("Train:", np.bincount(y_train, minlength=3))
print("Validation:", np.bincount(y_val, minlength=3))
print("Test:", np.bincount(y_test, minlength=3))

# ------------------------------------------------------------
# 5. PREPROCESSING
# ------------------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

# Scaled version for LR and SVM
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Unscaled version for RF and XGBoost
X_train_tree = X_train_imp
X_val_tree = X_val_imp
X_test_tree = X_test_imp

# ------------------------------------------------------------
# 6. METRICS
# ------------------------------------------------------------

def rps_multiclass(y_true, probs, n_classes=3):
    y_onehot = np.eye(n_classes)[y_true]
    cum_probs = np.cumsum(probs, axis=1)
    cum_true = np.cumsum(y_onehot, axis=1)
    return np.mean(np.sum((cum_probs - cum_true) ** 2, axis=1) / (n_classes - 1))


def expected_calibration_error(y_true, probs, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == y_true

    ece = 0.0
    bin_edges = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == n_bins - 1:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences >= lower) & (confidences < upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            acc_bin = np.mean(accuracies[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop_in_bin * abs(acc_bin - conf_bin)

    return ece


def evaluate_model(model, X, y):
    probs = model.predict_proba(X)
    preds = np.argmax(probs, axis=1)

    return {
        "Accuracy": accuracy_score(y, preds),
        "BalancedAccuracy": balanced_accuracy_score(y, preds),
        "LogLoss": log_loss(y, probs, labels=[0, 1, 2]),
        "RPS": rps_multiclass(y, probs),
        "ECE": expected_calibration_error(y, probs),
        "DrawRecall": recall_score(y, preds, labels=[1], average=None, zero_division=0)[0],
        "Pred_H": np.sum(preds == 0),
        "Pred_D": np.sum(preds == 1),
        "Pred_A": np.sum(preds == 2)
    }


def test_model(model, X_test_model, y_test, model_name):
    test_metrics = evaluate_model(model, X_test_model, y_test)

    print(f"\n=== FINAL TEST RESULTS: {model_name} ===")
    for key, value in test_metrics.items():
        print(f"{key}: {value}")

    test_probs = model.predict_proba(X_test_model)
    test_preds = np.argmax(test_probs, axis=1)

    print("\nPrediction distribution [H, D, A]:")
    print(np.bincount(test_preds, minlength=3))

    print("\nTrue distribution [H, D, A]:")
    print(np.bincount(y_test, minlength=3))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, test_preds, labels=[0, 1, 2]))

    return test_metrics, test_probs, test_preds

# ------------------------------------------------------------
# 7. LOGISTIC REGRESSION
# ------------------------------------------------------------

lr_grid = []

for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]:
    for class_weight in [None, "balanced"]:
        lr_grid.append({
            "name": f"LR_L2_C{C}_cw{class_weight}",
            "penalty": "l2",
            "solver": "lbfgs",
            "C": C,
            "class_weight": class_weight,
            "l1_ratio": None
        })

for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    for class_weight in [None, "balanced"]:
        lr_grid.append({
            "name": f"LR_L1_C{C}_cw{class_weight}",
            "penalty": "l1",
            "solver": "saga",
            "C": C,
            "class_weight": class_weight,
            "l1_ratio": None
        })

for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    for l1_ratio in [0.25, 0.5, 0.75]:
        for class_weight in [None, "balanced"]:
            lr_grid.append({
                "name": f"LR_EN_C{C}_l1{l1_ratio}_cw{class_weight}",
                "penalty": "elasticnet",
                "solver": "saga",
                "C": C,
                "class_weight": class_weight,
                "l1_ratio": l1_ratio
            })

lr_results = []
lr_models = {}

for params in lr_grid:
    model = LogisticRegression(
        penalty=params["penalty"],
        solver=params["solver"],
        C=params["C"],
        class_weight=params["class_weight"],
        l1_ratio=params["l1_ratio"],
        multi_class="multinomial",
        max_iter=5000,
        random_state=RANDOM_STATE
    )

    model.fit(X_train_scaled, y_train)

    train_metrics = evaluate_model(model, X_train_scaled, y_train)
    val_metrics = evaluate_model(model, X_val_scaled, y_val)

    lr_results.append({
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "LR",
        "Model": params["name"],
        "Penalty": params["penalty"],
        "C": params["C"],
        "ClassWeight": params["class_weight"],
        "L1Ratio": params["l1_ratio"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    lr_models[params["name"]] = model

lr_results_df = pd.DataFrame(lr_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== LR MODELS BY VALIDATION LOG LOSS ===")
display(lr_results_df.head(20))

best_lr_name = lr_results_df.loc[0, "Model"]
best_lr_model = lr_models[best_lr_name]

print("\nBest LR model by validation log loss:")
print(best_lr_name)

lr_test_metrics, lr_test_probs, lr_test_preds = test_model(
    best_lr_model,
    X_test_scaled,
    y_test,
    f"LR | {best_lr_name}"
)

# ------------------------------------------------------------
# 8. RANDOM FOREST
# ------------------------------------------------------------

rf_grid = [
    {
        "name": "RF_300_depth4_leaf5",
        "n_estimators": 300,
        "max_depth": 4,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": None
    },
    {
        "name": "RF_500_depth6_leaf5",
        "n_estimators": 500,
        "max_depth": 6,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": None
    },
    {
        "name": "RF_500_depth8_leaf10",
        "n_estimators": 500,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None
    },
    {
        "name": "RF_500_depth6_leaf5_balanced",
        "n_estimators": 500,
        "max_depth": 6,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": "balanced"
    }
]

rf_results = []
rf_models = {}

for params in rf_grid:
    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        class_weight=params["class_weight"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_tree, y_train)

    train_metrics = evaluate_model(model, X_train_tree, y_train)
    val_metrics = evaluate_model(model, X_val_tree, y_val)

    rf_results.append({
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "RF",
        "Model": params["name"],
        "N_estimators": params["n_estimators"],
        "MaxDepth": params["max_depth"],
        "MinSamplesLeaf": params["min_samples_leaf"],
        "MaxFeatures": params["max_features"],
        "ClassWeight": params["class_weight"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    rf_models[params["name"]] = model

rf_results_df = pd.DataFrame(rf_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== RF MODELS BY VALIDATION LOG LOSS ===")
display(rf_results_df)

best_rf_name = rf_results_df.loc[0, "Model"]
best_rf_model = rf_models[best_rf_name]

print("\nBest RF model by validation log loss:")
print(best_rf_name)

rf_test_metrics, rf_test_probs, rf_test_preds = test_model(
    best_rf_model,
    X_test_tree,
    y_test,
    f"RF | {best_rf_name}"
)

# ------------------------------------------------------------
# 9. XGBOOST
# ------------------------------------------------------------

XGB_PARAMS = {
    "tree_method": "hist"
}

xgb_grid = [
    {
        "name": "XGB_200_depth2_lr003",
        "n_estimators": 200,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0
    },
    {
        "name": "XGB_300_depth2_lr003_l1",
        "n_estimators": 300,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
        "reg_alpha": 0.5
    },
    {
        "name": "XGB_300_depth3_lr003",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },
    {
        "name": "XGB_300_depth3_lr005",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 7,
        "reg_lambda": 10.0,
        "reg_alpha": 0.5
    }
]

xgb_results = []
xgb_models = {}

for params in xgb_grid:
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **XGB_PARAMS
    )

    model.fit(X_train_tree, y_train)

    train_metrics = evaluate_model(model, X_train_tree, y_train)
    val_metrics = evaluate_model(model, X_val_tree, y_val)

    xgb_results.append({
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "XGB",
        "Model": params["name"],
        "N_estimators": params["n_estimators"],
        "MaxDepth": params["max_depth"],
        "LearningRate": params["learning_rate"],
        "Subsample": params["subsample"],
        "ColsampleBytree": params["colsample_bytree"],
        "MinChildWeight": params["min_child_weight"],
        "RegLambda": params["reg_lambda"],
        "RegAlpha": params["reg_alpha"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    xgb_models[params["name"]] = model

xgb_results_df = pd.DataFrame(xgb_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== XGB MODELS BY VALIDATION LOG LOSS ===")
display(xgb_results_df)

best_xgb_name = xgb_results_df.loc[0, "Model"]
best_xgb_model = xgb_models[best_xgb_name]

print("\nBest XGBoost model by validation log loss:")
print(best_xgb_name)

xgb_test_metrics, xgb_test_probs, xgb_test_preds = test_model(
    best_xgb_model,
    X_test_tree,
    y_test,
    f"XGB | {best_xgb_name}"
)

# ------------------------------------------------------------
# 10. CALIBRATED LINEAR SVM
# ------------------------------------------------------------

svm_grid = [
    {
        "name": "SVM_C0.01_cwNone",
        "C": 0.01,
        "class_weight": None
    },
    {
        "name": "SVM_C0.03_cwNone",
        "C": 0.03,
        "class_weight": None
    },
    {
        "name": "SVM_C0.1_cwNone",
        "C": 0.1,
        "class_weight": None
    },
    {
        "name": "SVM_C0.1_cwBalanced",
        "C": 0.1,
        "class_weight": "balanced"
    },
    {
        "name": "SVM_C0.3_cwNone",
        "C": 0.3,
        "class_weight": None
    }
]

svm_results = []
svm_models = {}

for params in svm_grid:
    base_svm = LinearSVC(
        C=params["C"],
        class_weight=params["class_weight"],
        random_state=RANDOM_STATE,
        max_iter=10000
    )

    model = CalibratedClassifierCV(
        estimator=base_svm,
        method="sigmoid",
        cv=3
    )

    model.fit(X_train_scaled, y_train)

    train_metrics = evaluate_model(model, X_train_scaled, y_train)
    val_metrics = evaluate_model(model, X_val_scaled, y_val)

    svm_results.append({
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "SVM",
        "Model": params["name"],
        "C": params["C"],
        "ClassWeight": params["class_weight"],
        "Calibration": "sigmoid_cv3",

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    svm_models[params["name"]] = model

svm_results_df = pd.DataFrame(svm_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== SVM MODELS BY VALIDATION LOG LOSS ===")
display(svm_results_df)

best_svm_name = svm_results_df.loc[0, "Model"]
best_svm_model = svm_models[best_svm_name]

print("\nBest SVM model by validation log loss:")
print(best_svm_name)

svm_test_metrics, svm_test_probs, svm_test_preds = test_model(
    best_svm_model,
    X_test_scaled,
    y_test,
    f"SVM | {best_svm_name}"
)

# ------------------------------------------------------------
# 11. FINAL SUMMARY
# ------------------------------------------------------------

summary_df = pd.DataFrame([
    {
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "LR",
        "BestModel": best_lr_name,
        **lr_test_metrics
    },
    {
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "RF",
        "BestModel": best_rf_name,
        **rf_test_metrics
    },
    {
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "XGB",
        "BestModel": best_xgb_name,
        **xgb_test_metrics
    },
    {
        "Experiment": EXPERIMENT,
        "FeatureSet": "structured_only",
        "ModelFamily": "SVM",
        "BestModel": best_svm_name,
        **svm_test_metrics
    }
]).sort_values("LogLoss").reset_index(drop=True)

print("\n================ FINAL STRUCTURED BASELINE SUMMARY ================")
display(summary_df)

summary_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_structured_only_model_summary.csv"
)

summary_df.to_csv(summary_path, index=False)

all_validation_results = pd.concat([
    lr_results_df,
    rf_results_df,
    xgb_results_df,
    svm_results_df
], axis=0).reset_index(drop=True)

validation_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_structured_only_validation_results.csv"
)

all_validation_results.to_csv(validation_path, index=False)

print("\nSaved summary to:")
print(summary_path)

print("\nSaved validation results to:")
print(validation_path)

# Calibration

In [ ]:
# ------------------------------------------------------------
# CALIBRATION PLOTS
# ------------------------------------------------------------

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def plot_multiclass_calibration(y_true, model_probs_dict, target_names, output_dir, experiment):
    """
    Plot one-vs-rest calibration curves for each class.
    Each class gets its own plot comparing all models.
    """

    n_classes = len(target_names)

    for class_id in range(n_classes):

        plt.figure(figsize=(7, 6))

        # Perfect calibration line
        plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

        y_binary = (y_true == class_id).astype(int)

        for model_name, probs in model_probs_dict.items():

            prob_class = probs[:, class_id]

            frac_pos, mean_pred = calibration_curve(
                y_binary,
                prob_class,
                n_bins=10,
                strategy="uniform"
            )

            plt.plot(
                mean_pred,
                frac_pos,
                marker="o",
                label=model_name
            )

        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed frequency")
        plt.title(f"Calibration plot: {target_names[class_id]}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        plot_path = os.path.join(
            output_dir,
            f"{experiment}_calibration_{target_names[class_id].replace(' ', '_').lower()}.png"
        )

        plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.show()

        print(f"Saved calibration plot to: {plot_path}")


model_probs_dict = {
    "Logistic Regression": lr_test_probs
}

plot_multiclass_calibration(
    y_true=y_test,
    model_probs_dict=model_probs_dict,
    target_names=target_names,
    output_dir=OUTPUT_DIR,
    experiment=EXPERIMENT
)

In [ ]:
# ============================================================
# CALIBRATION TABLE: BASELINE LR
# ============================================================

import numpy as np
import pandas as pd

class_names = ["Home win", "Draw", "Away win"]

def make_calibration_table(y_true, probs, model_name="Baseline LR", n_bins=10):
    rows = []

    bins = np.linspace(0, 1, n_bins + 1)

    for class_id, class_name in enumerate(class_names):

        y_binary = (y_true == class_id).astype(int)
        pred_prob = probs[:, class_id]

        bin_id = np.digitize(pred_prob, bins, right=False) - 1
        bin_id = np.clip(bin_id, 0, n_bins - 1)

        for b in range(n_bins):
            mask = bin_id == b

            if mask.sum() == 0:
                continue

            mean_predicted_probability = pred_prob[mask].mean()
            true_observed_probability = y_binary[mask].mean()

            rows.append({
                "model": model_name,
                "outcome": class_name,
                "probability_bin": f"{bins[b]:.1f}-{bins[b+1]:.1f}",
                "n_matches": mask.sum(),
                "predicted_probability": mean_predicted_probability,
                "true_probability": true_observed_probability,
                "calibration_error": true_observed_probability - mean_predicted_probability,
                "absolute_calibration_error": abs(true_observed_probability - mean_predicted_probability)
            })

    return pd.DataFrame(rows)


baseline_calibration_table = make_calibration_table(
    y_true=y_test,
    probs=lr_test_probs,
    model_name="Baseline LR",
    n_bins=10
)

baseline_calibration_table

,model,outcome,probability_bin,n_matches,predicted_probability,true_probability,calibration_error,absolute_calibration_error
0,Baseline LR,Home win,0.0-0.1,6,0.072192,0.166667,0.094474,0.094474
1,Baseline LR,Home win,0.1-0.2,11,0.164115,0.090909,-0.073206,0.073206
2,Baseline LR,Home win,0.2-0.3,19,0.259430,0.210526,-0.048904,0.048904
3,Baseline LR,Home win,0.3-0.4,12,0.349876,0.416667,0.066791,0.066791
4,Baseline LR,Home win,0.4-0.5,28,0.447889,0.535714,0.087825,0.087825
5,Baseline LR,Home win,0.5-0.6,27,0.542991,0.629630,0.086639,0.086639
6,Baseline LR,Home win,0.6-0.7,16,0.636286,0.437500,-0.198786,0.198786
7,Baseline LR,Home win,0.7-0.8,8,0.726482,0.750000,0.023518,0.023518
8,Baseline LR,Home win,0.8-0.9,3,0.864767,1.000000,0.135233,0.135233
9,Baseline LR,Draw,0.0-0.1,3,0.074757,0.000000,-0.074757,0.074757


## Probabilities

In [ ]:
# ============================================================
# EXPORT BASELINE LR TEST PROBABILITIES
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# ------------------------------------------------------------
# 1. Get probabilities and predictions from best baseline LR
# ------------------------------------------------------------

baseline_lr_probs = best_lr_model.predict_proba(X_test_scaled)
baseline_lr_preds = np.argmax(baseline_lr_probs, axis=1)

# ------------------------------------------------------------
# 2. Create match-level probability table
# ------------------------------------------------------------

baseline_lr_prob_table = df.loc[test_idx, [
    "MatchDatetime",
    "season",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "target"
]].copy()

baseline_lr_prob_table["true_label"] = baseline_lr_prob_table["target"].map(target_names)

baseline_lr_prob_table["baseline_pred"] = baseline_lr_preds
baseline_lr_prob_table["baseline_pred_label"] = baseline_lr_prob_table["baseline_pred"].map(target_names)

baseline_lr_prob_table["baseline_prob_home"] = baseline_lr_probs[:, 0]
baseline_lr_prob_table["baseline_prob_draw"] = baseline_lr_probs[:, 1]
baseline_lr_prob_table["baseline_prob_away"] = baseline_lr_probs[:, 2]

baseline_lr_prob_table["baseline_correct"] = (
    baseline_lr_prob_table["target"] == baseline_lr_prob_table["baseline_pred"]
)

baseline_lr_prob_table["baseline_confidence"] = baseline_lr_probs.max(axis=1)

sorted_probs = np.sort(baseline_lr_probs, axis=1)
baseline_lr_prob_table["baseline_margin"] = sorted_probs[:, -1] - sorted_probs[:, -2]

baseline_lr_prob_table = baseline_lr_prob_table.reset_index(drop=True)

# ------------------------------------------------------------
# 3. Display output
# ------------------------------------------------------------

print("\n=== BASELINE LR TEST PROBABILITY TABLE ===")
display(baseline_lr_prob_table.head(20))

print("\nPrediction distribution [H, D, A]:")
print(np.bincount(baseline_lr_preds, minlength=3))

print("\nTrue distribution [H, D, A]:")
print(np.bincount(y_test, minlength=3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, baseline_lr_preds, labels=[0, 1, 2]))

# ------------------------------------------------------------
# 4. Save output for manual comparison with factual/opinion
# ------------------------------------------------------------

baseline_lr_prob_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_baseline_lr_test_probabilities.csv"
)

baseline_lr_prob_table.to_csv(baseline_lr_prob_path, index=False)

print("\nSaved baseline LR test probabilities to:")
print(baseline_lr_prob_path)


=== BASELINE LR TEST PROBABILITY TABLE ===


,MatchDatetime,season,HomeTeam,AwayTeam,FTR,target,true_label,baseline_pred,baseline_pred_label,baseline_prob_home,baseline_prob_draw,baseline_prob_away,baseline_correct,baseline_confidence,baseline_margin
0,2024-02-17 15:00:00,2023_2024,Newcastle,Bournemouth,D,1,Draw,0,Home win,0.559232,0.216638,0.224129,False,0.559232,0.335103
1,2024-11-10 14:00:00,2024_2025,Tottenham,Ipswich,A,2,Away win,0,Home win,0.702250,0.158602,0.139148,False,0.702250,0.543649
2,2022-10-30 14:00:00,2022_2023,Arsenal,Nott'm Forest,H,0,Home win,0,Home win,0.843424,0.085316,0.071260,True,0.843424,0.758108
3,2022-10-01 12:30:00,2022_2023,Arsenal,Tottenham,H,0,Home win,0,Home win,0.536808,0.225616,0.237576,True,0.536808,0.299232
4,2024-10-05 12:30:00,2024_2025,Crystal Palace,Liverpool,A,2,Away win,2,Away win,0.188049,0.231171,0.580780,True,0.580780,0.349608
5,2022-09-17 15:00:00,2022_2023,Newcastle,Bournemouth,D,1,Draw,0,Home win,0.619447,0.193562,0.186991,False,0.619447,0.425884
6,2024-03-09 15:00:00,2023_2024,Bournemouth,Sheffield United,D,1,Draw,0,Home win,0.608205,0.204445,0.187350,False,0.608205,0.403759
7,2023-02-04 15:00:00,2022_2023,Man United,Crystal Palace,H,0,Home win,0,Home win,0.666886,0.174736,0.158378,True,0.666886,0.492150
8,2025-04-05 12:30:00,2024_2025,Everton,Arsenal,D,1,Draw,2,Away win,0.226422,0.261929,0.511650,False,0.511650,0.249721
9,2023-04-09 14:00:00,2022_2023,Leeds,Crystal Palace,A,2,Away win,0,Home win,0.430945,0.272664,0.296391,False,0.430945,0.134555



Prediction distribution [H, D, A]:
[88  0 42]

True distribution [H, D, A]:
[59 30 41]

Confusion matrix:
[[51  0  8]
 [22  0  8]
 [15  0 26]]

Saved baseline LR test probabilities to:
/content/drive/MyDrive/Speciale/ml_baseline_outputs/structured_with_odds_baseline_lr_test_probabilities.csv
